# Setup

In [ ]:
import sys
from pathlib import Path
import polars as pl
sys.path.append(str(Path().resolve().parent))

yellow_pages = pl.read_parquet(
    r"X:\Project-Shares\germart-lite\germart-papaio\lebensmittel2023.parquet").filter(pl.col("Strasse") != ""
).filter(
    pl.col("Branchen").is_in([
        "Lebensmittel",                     # Food products:    48509
        "Bio-Lebensmittel",                 # Organic foods:    379
        # "Lebensmitteltechnik",              # Food technology:  149
        # "Lebensmitteltransporte",           # Food transport:   46
        # "Lebensmitteluntersuchungen",       # Food testing:     35
        "Diätetische Lebensmittel",         # Dietary foods:    12
        "Asiatische Lebensmittel",          # Asian foods:      3
        # "Sachverständige: Lebensmittel",    # Food experts:     3
])
    )


panel_2023 = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\panel").filter(
    (pl.col("year") == 2023)
    & (pl.col("Shop Type") != "Drogeriemarkt")
    )

# Geocoding missing Yellow Pages entries

In [ ]:
from scripts.yellow_pages import yellow_pages_mtm as m

yellow_pages = yellow_pages.with_columns(
    pl.col("Koordinaten")
      .str.extract_all(r"-?\d+\.\d+")
      .alias("_coords")
).with_columns(
    pl.col("_coords").list.get(0, null_on_oob=True).cast(pl.Float64).alias("yp_lat_raw"),
    pl.col("_coords").list.get(1, null_on_oob=True).cast(pl.Float64).alias("yp_lon_raw"),
).drop("_coords")

from scripts.tdlinx.cleaning import (   # adjust import to wherever this module lives
    build_full_address, cleanup_csv, load_done_addresses,
    geocode_batch_photon, nominatim_retry, read_coord_cache, CSV_PATH,
)

# 1. Isolate the rows Koordinaten couldn't give us
missing = yellow_pages.filter(
    pl.col("yp_lat_raw").is_null() | pl.col("yp_lon_raw").is_null()
)
print(f"{missing.height:,} rows need geocoding")

# 2. build_full_address expects Straße/Plz/Ort column names — yellow_pages
#    uses different ones, so rename a copy rather than touching the module
missing_std = missing.rename({
    "Strasse": "Straße",
    "Postleitzahl": "Plz",
    # "Ort" already matches
})
missing_std = build_full_address(missing_std)

addresses_to_geocode = (
    missing_std["full_address"].drop_nulls().unique().to_list()
)
print(f"{len(addresses_to_geocode):,} unique geocodable addresses "
      f"({missing_std['full_address'].null_count():,} rows lack Straße/Plz/Ort entirely)")

# 3. Skip anything already resolved in the cache from a prior run
cleanup_csv(CSV_PATH)
done = load_done_addresses(CSV_PATH)
remaining = sorted(set(addresses_to_geocode) - done)
print(f"{len(remaining):,} left to actually geocode")

# 4. Phase 1 — Photon batch
if remaining:
    geocode_batch_photon(remaining, csv_path=CSV_PATH, batch_size=10)

# 5. Phase 2 — Nominatim retry for anything Photon couldn't resolve
coords = read_coord_cache(CSV_PATH)
still_failed = missing_std.join(coords, on="full_address", how="left").filter(
    pl.col("lat").is_null(), pl.col("full_address").is_not_null()
).select("full_address", "Straße", "Plz", "Ort")

if still_failed.height:
    nominatim_retry(still_failed, csv_path=CSV_PATH)

# 6. Pull the final cache and merge back into yellow_pages
coords = read_coord_cache(CSV_PATH)
yellow_pages = (
    yellow_pages.rename({"Strasse": "Straße", "Postleitzahl": "Plz"})
    .pipe(build_full_address, verbose=False)
    .join(coords, on="full_address", how="left", suffix="_geo")
    .with_columns(
        pl.coalesce(["yp_lat_raw", "lat"]).alias("yp_lat_raw"),
        pl.coalesce(["yp_lon_raw", "lon"]).alias("yp_lon_raw"),
    )
    .rename({"Straße": "Strasse", "Plz": "Postleitzahl"})
)

# Normalizing YP, PN, running matching

In [ ]:
yp_n = m.normalize_yellow_pages(
    yellow_pages,
    lat_col="yp_lat_raw",
    lon_col="yp_lon_raw",
)


pn_n = m.normalize_panel(panel_2023, year_col="year", year=2023)

edges = m.run_matching(yp_n, pn_n)                      # many-to-many edge table
# m.summarize(edges).show()
# m.block_report(edges).show(10)                          # crowded addresses
# m.review_queue(edges).show(10)                          # ambiguity, grouped by block

xw   = m.crosswalk(edges, min_confidence="medium")      # kept yp<->TDLinx pairs
one  = m.mutual_best_only(edges)                        # conservative 1:1 core


# Output explaination:
- `edges`: One row per candidate pair
    - `yp_id`: unique id per yellow pages row
    - `TDLinx`: Panel store id, `Null` if yellow page row did not match any panel store 
    - `block_id`: addresses; same address in yellow pages and panel translates into same `block_id` in `edges`
    - `block_kind`: rank of matching: `addr_hnr` matched everything, incl. house number, `addr_ort` matched Ort, but not Postleitzahl, `addr_str` matched street, and Postleitzahl.
- 

In [ ]:
# none = edges.filter(pl.col("block_kind") == "none").join(
#     yp_n.select("yp_id", "yp_name_n", "yp_gruppe"), on="yp_id", how="left"
# )
# none.group_by("yp_gruppe").len().sort("len", descending=True).head(30)

In [ ]:
# matched_pn = xw.select("TDLinx").unique()
# orphans = pn_n.join(matched_pn, on="TDLinx", how="anti")
# orphans.group_by("pn_gruppe", "pn_shop_type").len().sort("len", descending=True)

In [ ]:
# edges.filter(pl.col("edge_class") == "gruppe_conflict") \
#      .group_by("yp_gruppe", "pn_gruppe").len().sort("len", descending=True)

In [ ]:
# print(pn_n.height)                                    # should be ~4,400 lower
# edges.group_by("block_kind").len().sort("len", descending=True)
# edges.group_by("edge_class").len().sort("len", descending=True)   # dominated should have collapsed

In [ ]:
# yp_unmatched = edges.filter(pl.col("block_kind") == "none")
# print(yp_unmatched.height)   # 8,182

In [ ]:
# yp_unmatched = yp_n.join(
#     edges.filter(pl.col("block_kind") == "none").select("yp_id"),
#     on="yp_id", how="semi",
# )

In [ ]:
yp_n.filter(
    pl.col("yp_gruppe").is_null()
).group_by(
    pl.col("yp_name_n")
    ).len().sort(
        "len", descending=True
    ).filter(
        pl.col("len")>10,
    # ).select(
        # "yp_name_n"
        ).show(None
               )

In [ ]:
unknown = (edges.filter(pl.col("yp_gruppe").is_null()
                        & pl.col("TDLinx").is_not_null()
                        & (pl.col("block_kind") == "addr_hnr"))
                .join(yp_n.select("yp_id", "yp_name_n"), on="yp_id", how="left"))

verdict = (unknown.group_by("yp_name_n")
    .agg(pl.len().alias("n"),
         pl.col("pn_gruppe").drop_nulls().mode().first().alias("owner"),
         (pl.col("pn_gruppe") == pl.col("pn_gruppe").drop_nulls().mode().first())
            .mean().round(2).alias("purity"),
         pl.col("pn_gruppe").drop_nulls().n_unique().alias("n_distinct"))
    .filter(pl.col("n") >= 3).sort("n", descending=True))
verdict.show(None)